# Video 07 - The golden rules of tools

**You are here:** Video 7 of 13 - the end of the Tools phase (see `../you-are-here.svg`).

**The one idea:** tool messages have strict rules. Some models enforce them, some quietly don't - so **write to the strictest rules**. And what you put in the context window decides cost *and* correctness.

**By the end you can:**
1. Follow the four golden rules - and know what breaks when you don't.
2. Write one helper that adds tool results correctly.
3. Design tools that give the model only what it needs (context engineering).

**Before you start:** attach this notebook to **Serverless**.

In [0]:
import json
from databricks.sdk import WorkspaceClient

client = WorkspaceClient().serving_endpoints.get_open_ai_client()
MODEL = "databricks-meta-llama-3-3-70b-instruct"
OTHER_MODEL = "databricks-claude-haiku-4-5"

def get_weather(city: str) -> str:
    return {"perth": "sunny, 31C", "london": "rainy, 12C", "tokyo": "cloudy, 18C"}.get(city.lower(), "unknown city")

WEATHER_TOOL = {"type": "function", "function": {
    "name": "get_weather", "description": "Get the current weather for a city.",
    "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}}

def try_call(model, messages, tools=[WEATHER_TOOL]):
    """Call the model and report OK or the error, instead of crashing."""
    try:
        reply = client.chat.completions.create(model=model, messages=messages, tools=tools).choices[0].message
        return "OK    " + (reply.content or "(asked for another tool)")[:45]
    except Exception as e:
        return "ERROR " + str(e)[:60]

print("Connected.")

## Part 1 - The four golden rules
(diagram: `../diagrams/01-four-golden-rules.svg`)

| # | Rule | In code |
| --- | --- | --- |
| 1 | **Request before result** | append the assistant message with `tool_calls` first |
| 2 | **Match the id** | `"tool_call_id": call.id` |
| 3 | **Results are strings** | `json.dumps(result)` for dicts and lists |
| 4 | **Answer every call** | one `tool` message per call |

Let's break each rule on purpose and see what each model does.

In [0]:
QUESTION = [{"role": "user", "content": "Should I bring an umbrella in London today?"}]

requests = {}
for model in [MODEL, OTHER_MODEL]:
    msg = client.chat.completions.create(model=model, messages=QUESTION, tools=[WEATHER_TOOL]).choices[0].message
    requests[model] = msg
    print(f"{model}: wants {msg.tool_calls[0].function.name}({msg.tool_calls[0].function.arguments}) id={msg.tool_calls[0].id[:20]}...")

### Break rules 1 to 3
For each model we send back five versions of the conversation.

**Predict first:** which broken versions will each model accept?

In [0]:
for model, msg in requests.items():
    call = msg.tool_calls[0]
    request = msg.model_dump(exclude_none=True)
    versions = {
        "correct":                  QUESTION + [request, {"role": "tool", "tool_call_id": call.id, "content": "rainy, 12C"}],
        "rule 1 broken (no request)": QUESTION + [{"role": "tool", "tool_call_id": call.id, "content": "rainy, 12C"}],
        "rule 2 broken (wrong id)": QUESTION + [request, {"role": "tool", "tool_call_id": "call_wrong", "content": "rainy, 12C"}],
        "rule 3 broken (dict)":     QUESTION + [request, {"role": "tool", "tool_call_id": call.id, "content": {"sky": "rainy", "temp": 12}}],
        "rule 3 fixed (json.dumps)": QUESTION + [request, {"role": "tool", "tool_call_id": call.id, "content": json.dumps({"sky": "rainy", "temp": 12})}],
    }
    print(f"=== {model} ===")
    for name, messages in versions.items():
        print(f"  {name:28} {try_call(model, messages)}")
    print()

## What could go wrong? Lenient models hide bugs
(diagram: `../diagrams/02-strict-vs-lenient.svg`)

In our tests Llama **silently accepted** rule 1 and rule 2 being broken; Claude rejected them. If you only ever test on a lenient model, your code passes every test - and crashes the day the model changes. **Write to the strictest rules.**

### Break rule 4: answer every call
Rule 4 needs a model that asks for **several** tools at once. Claude does; the Databricks Llama endpoint returns only one tool call per reply (that's why Llama asked for Perth, then Tokyo, in Video 06).

**Predict first:** what happens if we answer only the first of Claude's two calls?

In [0]:
two_cities = [{"role": "user", "content": "What's the weather in Perth and in Tokyo?"}]
msg = client.chat.completions.create(model=OTHER_MODEL, messages=two_cities, tools=[WEATHER_TOOL]).choices[0].message
print("Claude asked for", len(msg.tool_calls), "tool calls:", [c.function.arguments for c in msg.tool_calls])

first = msg.tool_calls[0]
only_first = two_cities + [
    msg.model_dump(exclude_none=True),
    {"role": "tool", "tool_call_id": first.id, "content": get_weather(**json.loads(first.function.arguments))},
]
print("Answered only the first call ->", try_call(OTHER_MODEL, only_first))

## One helper that follows all four rules
Write the rules down **once**, in code - and never build tool messages by hand again.

In [0]:
def add_tool_results(messages, msg, functions):
    """Run every tool the model asked for and append the results - following all four golden rules."""
    messages.append(msg.model_dump(exclude_none=True))                          # rule 1: request first
    for call in msg.tool_calls:                                                  # rule 4: every call
        args = json.loads(call.function.arguments or "{}")
        result = functions[call.function.name](**args)
        content = result if isinstance(result, str) else json.dumps(result)      # rule 3: strings only
        messages.append({"role": "tool", "tool_call_id": call.id, "content": content})  # rule 2: matching id

def ask_with_tools(question, tools, functions, model=MODEL):
    """One tool round using the safe helper. Returns (answer, input tokens of the second call)."""
    messages = [{"role": "user", "content": question}]
    msg = client.chat.completions.create(model=model, messages=messages, tools=tools).choices[0].message
    if not msg.tool_calls:
        return msg.content, 0
    add_tool_results(messages, msg, functions)
    response = client.chat.completions.create(model=model, messages=messages, tools=tools)
    return response.choices[0].message.content, response.usage.prompt_tokens

answer, _ = ask_with_tools("What's the weather in Perth and in Tokyo?", [WEATHER_TOOL], {"get_weather": get_weather}, OTHER_MODEL)
print("Claude, two calls handled correctly:\n", answer)

## Part 2 - Context engineering
Everything the model sees lives in the **context window**: system prompt + tool menu + history + **tool results**. Tool results are often the biggest part (diagram: `../diagrams/03-context-budget.svg`).

Same question, two tool designs:
* `get_all_orders()` - dump every order and let the model work it out.
* `get_order_total(customer_id, month)` - do the work in Python, return one small result.

**Predict first:** which gives the right answer? How many input tokens will each use?

In [0]:
import random
random.seed(11)   # same fake data every run

ORDERS = [
    {
        "order_id": 1000 + i,
        "customer_id": random.choice([7, 42, 99]),
        "date": f"2026-{random.randint(1, 6):02d}-{random.randint(1, 28):02d}",
        "status": random.choice(["delivered", "delivered", "shipped", "cancelled"]),
        "items": random.randint(1, 5),
        "total": round(random.uniform(10, 400), 2),
        "shipping_address": "12 Example Street, Perth WA 6000",
        "notes": "Leave at front door if nobody is home.",
    }
    for i in range(400)
]

march_42 = [o["total"] for o in ORDERS if o["customer_id"] == 42 and o["date"].startswith("2026-03") and o["status"] != "cancelled"]
TRUTH = round(sum(march_42), 2)
print(f"{len(ORDERS)} orders | customer 42, March 2026, not cancelled: {len(march_42)} orders, true total ${TRUTH}")

In [0]:
def get_all_orders():
    return ORDERS                                               # everything, every field

def get_order_total(customer_id: int, month: str):
    rows = [o for o in ORDERS if o["customer_id"] == customer_id and o["date"].startswith(month) and o["status"] != "cancelled"]
    return {"customer_id": customer_id, "month": month, "orders_counted": len(rows), "total": round(sum(o["total"] for o in rows), 2)}

BIG_TOOL = {"type": "function", "function": {
    "name": "get_all_orders", "description": "Return every order in the system with all fields.",
    "parameters": {"type": "object", "properties": {}}}}
NARROW_TOOL = {"type": "function", "function": {
    "name": "get_order_total", "description": "Total spent by one customer in one month, excluding cancelled orders.",
    "parameters": {"type": "object", "properties": {
        "customer_id": {"type": "integer"},
        "month": {"type": "string", "description": "YYYY-MM, e.g. 2026-03"}},
        "required": ["customer_id", "month"]}}}

print(f"get_all_orders() result: {len(json.dumps(get_all_orders())):,} characters")
print(f"get_order_total() result: {json.dumps(get_order_total(42, '2026-03'))}")

In [0]:
ORDER_QUESTION = "How much did customer 42 spend in March 2026, not counting cancelled orders?"
results = {}

for model in [MODEL, OTHER_MODEL]:
    for name, tool, functions in [
        ("big dump", BIG_TOOL, {"get_all_orders": get_all_orders}),
        ("narrow tool", NARROW_TOOL, {"get_order_total": get_order_total}),
    ]:
        answer, tokens = ask_with_tools(ORDER_QUESTION, [tool], functions, model)
        results[(model, name)] = tokens
        correct = f"{TRUTH:,.2f}" in (answer or "") or f"{TRUTH:.2f}" in (answer or "")
        print(f"{model:40} {name:12} input tokens={tokens:6,} | correct: {correct} | ...{(answer or '')[-70:]!r}")

## Cost meter: big dumps are expensive
The tool result is sent as input on the next call - and on **every** call after that if the conversation continues. Prices below are **example numbers**.

In [0]:
PRICE_PER_1M_INPUT = 0.50   # EXAMPLE price in USD - replace with your real rate

big = results[(MODEL, "big dump")]
narrow = results[(MODEL, "narrow tool")]
for name, tokens in [("big dump", big), ("narrow tool", narrow)]:
    per_q = tokens / 1e6 * PRICE_PER_1M_INPUT
    print(f"{name:12}: {tokens:6,} input tokens -> ${per_q:.5f} per question -> ${per_q * 10_000:,.2f} per 10,000 questions")
print(f"\nThe big dump uses {big / narrow:.0f}x more input tokens.")

## Recap
* **Four golden rules:** request before result, match the id, results are strings, answer every call.
* **Lenient models hide bugs** - write to the strictest rules, and use one helper (`add_tool_results`).
* The Databricks Llama endpoint returns **one tool call per reply**; Claude can return several.
* **Context engineering:** do the work in code, return small exact results. Big dumps cost more *and* can give wrong answers.

## Practise
* [exercises.md](../exercises.md) - 3 exercises + bonus (about 15 minutes). Add your code below this cell.
* [quiz.md](../quiz.md) - 5 questions.
* Solutions: `07-tool-rules-solutions` (try first!).

**Next:** Video 08 - The agent loop. Everything so far, in a loop: your first real agent.